In [23]:
from __future__ import annotations
from typing import Self

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import keras
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score

In [2]:
def load_and_clean_data() -> tuple[tuple[np.ndarray, ...], ...]:
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    x_train //= 255
    x_test //= 255
    return (x_train, y_train), (x_test, y_test)


(x_train, y_train), (x_test, y_test) = load_and_clean_data()

In [18]:
x_train_flat = x_train.reshape(x_train.shape[0], 
                               x_train.shape[1] * x_train.shape[2])
x_test_flat = x_test.reshape(x_test.shape[0], 
                             x_test.shape[1] * x_test.shape[2])

In [15]:
x_train_flat.shape

(60000, 784)

In [25]:
x_train.reshape(x_train.shape[0], -1).shape

(60000, 784)

In [32]:
class MultiLogisticRegression:
    def __init__(self) -> None:
        self._models = [LogisticRegression() for _ in range(10)]
        return
    
    def fit(self, X: np.ndarray, y: np.ndarray) -> Self:
        X_flat = X.reshape(X.shape[0], -1)
        for idx, model in enumerate(self._models):
            y_i = (y == idx).astype(int)
            model.fit(X_flat, y_i)
        return

    def predict(self, X: np.ndarray) -> np.ndarray:
        X_flat = X.reshape(X.shape[0], -1)
        y_pred_arrays: list[np.ndarray] = []
        for model in self._models:
            y_pred_arrays.append(model.predict_proba(X_flat))
        y_pred = np.concat(y_pred_arrays, axis = 1)
        return y_pred.argmax(axis = 1)

In [33]:
logistic_model = MultiLogisticRegression()
logistic_model.fit(x_train, y_train)
y_pred = logistic_model.predict(x_test)
accuracy_score(y_test, y_pred)

0.0511

In [75]:
X_flat = x_test.reshape(x_test.shape[0], -1)
y_pred_arrays = []
for model in logistic_model._models:
    y_pred_i = model.predict_proba(X_flat)
    y_pred_arrays.append(y_pred_i[:, 0].reshape(-1, 1))

y_pred = np.concat(y_pred_arrays, axis = 1).argmax(axis = 1)
y_pred

array([1, 7, 0, ..., 4, 1, 0])

In [76]:
accuracy_score(y_test, y_pred)

0.0208

In [78]:
28*28

784

In [82]:
def plot_history(history: keras.callbacks.History) -> None:
    fig, ax = plt.subplots()
    colors = ["steelblue", "orange", "springgreen", "slateblue"]
    
    metric_names = list(history.history.keys())
    metric_values = list(history.history.values())
    epochs = list(range(1, len(metric_values) + 1))

    for i in range(len(metric_values)):
        ax.plot(epochs, metric_values[i], 
                color = colors[i], 
                label = metric_names[i])
    
    ax.legend()
    ax.set_xlabel("Epochs")
    ax.set_ylabel("Metric Value")
    plt.show()

In [83]:
model = keras.Sequential([
    keras.layers.Flatten(), 
    keras.layers.Dense(784, activation = "relu"), 
    keras.layers.Dense(128, activation = "relu"), 
    keras.layers.Dense(64, activation = "relu"), 
    keras.layers.Dense(10, activation = "softmax"), 
])

callback = keras.callbacks.EarlyStopping(patience = 5)

model.compile(optimizer = "adam", 
              loss = "SparseCategoricalCrossentropy", 
              metrics = ["accuracy"])
history = model.fit(x_train, 
                    y_train, 
                    epochs = 30, 
                    validation_split = 0.2, 
                    callbacks = [callback])

plot_history(history)

Epoch 1/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 1.8937 - val_loss: 1.6638
Epoch 2/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 23s 16ms/step - loss: 1.6025 - val_loss: 1.6329
Epoch 3/30


KeyboardInterrupt: 